# Amazon Review 2023
Amazon Reviews dataset is large-scale dataset collected in 2023 by McAuley Lab, and it includes rich features such as:
- User Reviews (ratings, text, helpfulness votes, etc.);
- Item Metadata (descriptions, price, raw image, etc.);
- Links (user-item / bought together graphs).

Related Information
- HP: https://amazon-reviews-2023.github.io/
- paper: [Bridging Language and Items for Retrieval and Recommendation](https://arxiv.org/abs/2403.03952)


In [1]:
%load_ext autoreload

In [ ]:
%autoreload 2

import pathlib

from torch_geometric.data import HeteroData

from ml_sandbox_libs.data.amazon_reviews_dataset import (
    AmazonReviewsSeqRecDataModule,
    bipartite_graph_preprocess_dataset,
    fetch_dataset,
    fetch_metadata,
)

In [3]:
dataset_dict = fetch_dataset(category="Video_Games", dataset_type="0core_timestamp_w_his")
df = dataset_dict["train"].to_polars()
df.head()

2025-07-19 14:25:04.616 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_dataset:32 - Fetching Amazon Reviews 2023 dataset


user_id,parent_asin,rating,timestamp,history
str,str,str,str,str
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07SRWRH5D""","""5.0""","""1587051114941""",""""""
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07DK1H3H5""","""4.0""","""1608186804795""","""B07SRWRH5D"""
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""","""B07MFMFW34""","""5.0""","""1490877431000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B00HUWA45W""","""5.0""","""1427591932000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B0BCHWZX95""","""5.0""","""1577637634017""","""B00HUWA45W"""


In [4]:
print("Dataset Size")
print(
    f"train: {len(dataset_dict['train'])}, valid: {len(dataset_dict['valid'])}, test: {len(dataset_dict['test'])}"
)

Dataset Size
train: 3847041, valid: 344592, test: 363867


The dataset schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews

| Field            | Type     | Explanation                                                                                                                                                                               |
|------------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| user_id          | str      | ID of the reviewer                                                                                                                                                                       |
| parent_asin      | str      | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. **Please use parent ID to find product meta.** |
| rating           | float    | Rating of the product (from 1.0 to 5.0).                                                                                                                                                   |
| timestamp        | int      | Time of the review (unix time)                                                                                                                                                           |
| history | str     | parent_asin list which was bought by user before. The separator is ' '                                                                                                                                                               |

In [5]:
metadata_dataset = fetch_metadata(category="Video_Games")
metadata_df = metadata_dataset.to_polars().select(["parent_asin", "title", "categories"])
metadata_df.head(5)

2025-07-19 14:25:08.786 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_metadata:52 - Fetching Amazon Reviews 2023 metadata


parent_asin,title,categories
str,str,list[str]
"""B000FH0MHO""","""Dash 8-300 Professional Add-On""","[""Video Games"", ""PC"", ""Games""]"
"""B00069EVOG""","""Phantasmagoria: A Puzzle of Fl…","[""Video Games"", ""PC"", ""Games""]"
"""B00Z9TLVK0""","""NBA 2K17 - Early Tip Off Editi…","[""Video Games"", ""PlayStation 4"", ""Games""]"
"""B07SZJZV88""","""Nintendo Selects: The Legend o…","[""Video Games"", ""Legacy Systems"", … ""Games""]"
"""B002WH4ZJG""","""Thrustmaster Elite Fitness Pac…","[""Video Games"", ""Legacy Systems"", … ""Fitness Accessories""]"


In [6]:
print(f"Parent Asin Size: {len(metadata_df)}")

Parent Asin Size: 137269


The metadata schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata

| Field           | Type   | Explanation                                                                                                                                                             |
|-----------------|--------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| parent_asin     | str    | Parent ID of the product.                                                                                                                                                |
| title           | str    | Name of the product.                                                                                                                                                     |
| categories      | list   | Hierarchical categories of the product.                                                                                                                                  |


In [6]:
datamodule = AmazonReviewsSeqRecDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=4,
    max_seq_len=5,
    neg_sample_size=2,
    sampling_val_test=True,
    eval_negative_sample_size=10,
    filter_no_history=False,
)

datamodule.prepare_data()
datamodule.setup(stage="fit")
train_dataloader = datamodule.train_dataloader()
batch = next(iter(train_dataloader))

2025-07-19 14:20:42.553 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:prepare_data:371 - Preprocessed dataset not found
2025-07-19 14:20:42.554 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_dataset:32 - Fetching Amazon Reviews 2023 dataset
2025-07-19 14:20:43.930 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.common:fetch_metadata:52 - Fetching Amazon Reviews 2023 metadata
2025-07-19 14:20:49.119 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:seq_rec_preprocess_dataset:145 - Preprocessing the train dataset
2025-07-19 14:20:55.491 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:seq_rec_preprocess_dataset:147 - Preprocessing the val dataset
2025-07-19 14:20:56.000 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.seq_rec:seq_rec_preprocess_dataset:149 - Preprocessing the test dataset
python(49338) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(51388) 

In [7]:
datamodule.train_df.head(5)

user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,history,history_index,history_category,history_category_index
str,i64,str,i64,str,i64,f64,i64,list[str],list[i64],list[str],list[i64]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216183,"""B07SRWRH5D""",30010,"""Video Games/PlayStation 4/Game…",164,5.0,1587051114941,[],[],[],[]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216183,"""B07DK1H3H5""",27742,"""Video Games/PC/Games""",146,4.0,1608186804795,"[""B07SRWRH5D""]",[30010],"[""Video Games/PlayStation 4/Games""]",[164]
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""",1576344,"""B07MFMFW34""",29082,"""Video Games/PC/Games""",146,5.0,1490877431000,[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961353,"""B00HUWA45W""",19294,"""Video Games/Xbox One/Accessori…",177,5.0,1427591932000,[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961353,"""B0BCHWZX95""",33733,"""Video Games/Nintendo Switch/Ac…",121,5.0,1577637634017,"[""B00HUWA45W""]",[19294],"[""Video Games/Xbox One/Accessories""]",[177]


In [8]:
batch.user_index, batch.pos_item_index, batch.neg_item_indexes, batch.item_history

(tensor([1834524, 1638614]),
 tensor([ 4495, 11670]),
 tensor([[21180,  5834],
         [17201, 20327]]),
 tensor([[    0,     0,     0,     0,  2940],
         [ 2588,  7129,  5972, 16504,  9341]]))

## Bipartite Graph

In [45]:
(
    all_df,
    user2index,
    item2index,
    category2index,
    item_index_2_category_index,
) = bipartite_graph_preprocess_dataset(dataset_dict=dataset_dict, metadata=metadata_dataset)

2025-07-19 17:30:10.088 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset.bipartite_graph:bipartite_graph_preprocess_dataset:83 - Preprocessing the dataset for bipartite graph


In [46]:
all_df.head(5)

split,user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,num_ratings
str,str,i64,str,i64,str,i64,f64,i64,u32
"""train""","""AHCQT34DYQABEWQNREWCADHVTP3Q""",1758608,"""B017V6YVDC""",22580,"""Video Games/Online Game Servic…",133,5.0,1485437133000,1
"""train""","""AHHZ4PKIW2HA7DNNHCKJ53XJVLJQ""",1846584,"""B00JDP1AWU""",19621,"""Video Games/PC/Games""",146,5.0,1417294247000,1
"""train""","""AG5I3XFMYBZCYWMUHQERG2JX4HVA""",1131736,"""B08392WSFZ""",31315,"""Video Games/Legacy Systems/Nin…",20,5.0,1494724941000,1
"""train""","""AFTVQDHKFI7ZSXSV4KXEOGSBOZYA""",971375,"""B000Q09NGQ""",7824,"""Video Games/Legacy Systems/Nin…",60,5.0,1370209538000,1
"""train""","""AEPECS7XLVK2SGKYJUVR2FFHZMUA""",357494,"""B000OYMYZQ""",7724,"""Video Games/Legacy Systems/Xbo…",1,5.0,1338492967000,1


In [ ]:
import numpy as np
import polars as pl
import torch
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.sampler import NegativeSampling

user_index = torch.as_tensor(sorted(user2index.values()), dtype=torch.int64)

In [49]:
item_df = pl.from_dict({"item_index": item2index.values()})
item2category_df = pl.from_dict(
    {
        "item_index": item_index_2_category_index.keys(),
        "category": item_index_2_category_index.values(),
    }
)
item_df = item_df.join(item2category_df, on="item_index", validate="m:1").sort("item_index")
item_index = item_df["item_index"].to_torch()
category_index = item_df["category"].to_torch()

In [55]:
all_df.filter(pl.col("split") == "train")

split,user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,num_ratings
str,str,i64,str,i64,str,i64,f64,i64,u32
"""train""","""AHCQT34DYQABEWQNREWCADHVTP3Q""",1758608,"""B017V6YVDC""",22580,"""Video Games/Online Game Servic…",133,5.0,1485437133000,1
"""train""","""AHHZ4PKIW2HA7DNNHCKJ53XJVLJQ""",1846584,"""B00JDP1AWU""",19621,"""Video Games/PC/Games""",146,5.0,1417294247000,1
"""train""","""AG5I3XFMYBZCYWMUHQERG2JX4HVA""",1131736,"""B08392WSFZ""",31315,"""Video Games/Legacy Systems/Nin…",20,5.0,1494724941000,1
"""train""","""AFTVQDHKFI7ZSXSV4KXEOGSBOZYA""",971375,"""B000Q09NGQ""",7824,"""Video Games/Legacy Systems/Nin…",60,5.0,1370209538000,1
"""train""","""AEPECS7XLVK2SGKYJUVR2FFHZMUA""",357494,"""B000OYMYZQ""",7724,"""Video Games/Legacy Systems/Xbo…",1,5.0,1338492967000,1
…,…,…,…,…,…,…,…,…,…
"""train""","""AEWMGQIGXREMAMDAEP6S3DEE6WRA""",479273,"""B07KPGMK62""",1,"""Video Games/PC/Accessories/Hea…",145,1.0,1617214014034,1
"""train""","""AH6YLSKU6C5LEH4NSV4EA6C4OYTQ""",1695426,"""B00066LGEM""",5372,"""Video Games/Legacy Systems/Pla…",64,1.0,1560652150439,1
"""train""","""AHHSI6H2ZQUZUVQDUA2PJNU4XUIQ""",1843043,"""B09CD55646""",33221,"""Video Games/Legacy Systems/Nin…",53,1.0,1575205229288,1


In [91]:
def create_bipartite_graph(
    split: str,
    all_df: pl.DataFrame,
    user2index: dict[str, int],
    item2index: dict[str, int],
    item_index_2_category_index: dict[int, int],
):
    user_index = torch.as_tensor(sorted(user2index.values()), dtype=torch.int64)
    item_df = pl.from_dict({"item_index": item2index.values()})
    item2category_df = pl.from_dict(
        {
            "item_index": item_index_2_category_index.keys(),
            "category": item_index_2_category_index.values(),
        }
    )
    item_df = item_df.join(item2category_df, on="item_index", validate="m:1").sort("item_index")
    item_index = item_df["item_index"].to_torch()
    category_index = item_df["category"].to_torch()

    match split:
        case "train":
            df = all_df.filter(pl.col("split") == "train")
        case "valid":
            df = all_df.filter(pl.col("split").is_in(["train", "valid"]))
        case "test":
            df = all_df.filter(pl.col("split").is_in(["train", "valid", "test"]))
        case _:
            raise ValueError(f"Invalid split: {split}")

    edge_index = torch.as_tensor(
        np.ascontiguousarray(df["user_index", "item_index"].to_numpy().T), dtype=torch.long
    )
    edge_label_index = torch.as_tensor(
        np.ascontiguousarray(
            df.filter(pl.col("split") == split)["user_index", "item_index"].to_numpy().T
        ),
        dtype=torch.long,
    )
    data = HeteroData(
        {
            "user": {"x": user_index.unsqueeze(-1), "user_index": user_index},
            "item": {
                "x": item_index.unsqueeze(-1),
                "item_index": item_index,
                "category_index": category_index,
            },
            ("user", "rates", "item"): {
                "edge_index": edge_index,
                "edge_label_index": edge_label_index,
            },
        }
    )
    return data

In [92]:
train_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)
val_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)

transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
train_data = transform(train_data)
val_data = transform(val_data)

In [ ]:
train_data

HeteroData(
  user={
    x=[2149655, 1],
    user_index=[2149655],
  },
  item={
    x=[34153, 1],
    item_index=[34153],
    category_index=[34153],
  },
  (user, rates, item)={
    edge_index=[2, 4191633],
    edge_label_index=[2, 344592],
  }
)

In [ ]:
neg_sampling = NegativeSampling(mode="triplet", amount=3)

loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[10, 5],
    batch_size=2,
    edge_label_index=(
        ("user", "rates", "item"),
        train_data["user", "rates", "item"].edge_label_index,
    ),
    edge_label=None,
    neg_sampling=neg_sampling,
    shuffle=True,
)

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [94]:
train_data["user", "rates", "item"].edge_index

tensor([[1758607, 1846583, 1131735,  ..., 1843042,   54603, 1844944],
        [  22579,   19620,   31314,  ...,   33220,   20819,   15244]])

In [79]:
train_data.keys()

['user_index',
 'edge_label_index',
 'edge_index',
 'item_index',
 'category_index',
 'x']

In [73]:
val_data["user"].x.size()

torch.Size([2])

In [ ]:
train_df = all_df.filter(pl.col("split") == "train")
user_item_edge_index = train_df["user_index", "item_index"].to_torch(dtype=pl.Int64)

data = HeteroData(
    {
        "user": {"x": user_index, "user_index": user_index},
        "item": {"x": user_index, "item_index": item_index, "category_index": category_index},
        ("user", "rates", "item"): {"edge_index": user_item_edge_index},
    }
)
transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
data = transform(data)

In [54]:
data = HeteroData(
    {
        "user": {"user_index": user_index},
        "item": {"item_index": item_index, "category_index": category_index},
        ("user", "rates", "item"): {"edge_index": user_item_edge_index},
    }
)

In [43]:
data

HeteroData(
  user={ user_index=[2149656] },
  item={
    item_index=[34154],
    category_index=[34154],
  },
  (user, rates, item)={ edge_index=[4555500, 2] }
)